In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'https://bh.rdm.yzwlab.com/'
idp_name_1 = None
idp_username_1 = None
idp_password_1 = None
rdm_project_name = 'TEST-{}'.format(datetime.now().strftime('%Y%m%d-%H%M%S'))
target_storage_name = 'NII Storage'
target_storage_id = 'osfstorage'
delete_project = True
default_result_path = None
close_on_fail = False
transition_timeout = 10000

paper_doi = '10.52825/cordi.v1i.260'
paper_title_ja = 'オントロジー技術を用いたNII RDCアプリケーションプロファイルの開発'
paper_title_en = 'Toward the Development of NII RDC Application Profile Using Ontology Technology'
paper_publication_month = '2023-09'
paper_journal_name_ja = '研究データ基盤会議論文集'
paper_journal_name_en = 'Proceedings of the Conference on Research Data Infrastructure'
paper_journal_volume = '1'
paper_journal_issue = '1'
paper_page_start = '1'
paper_page_end = '8'
paper_pdf_url = 'https://www.tib-op.org/ojs/index.php/CoRDI/article/download/260/468'
paper_repo_name_ja = '研究データ基盤会議論文集'
paper_repo_name_en = 'Proceedings of the Conference on Research Data Infrastructure'
paper_description_ja = '国立研究データクラウドにおけるアプリケーションプロファイル開発の取り組みを紹介する論文です。'
paper_description_en = ('Application profile development for research data platforms is discussed in this paper.')

metadata_search_keyword = paper_title_ja
paper_authors = [
    {
        'number': '70921876',
        'name_ja': {'last': '南山', 'middle': '', 'first': '泰行'},
        'name_en': {'last': 'Minamiyama', 'middle': '', 'first': 'Yasuyuki'},
        'affiliation_ja': '国立情報学研究所',
        'affiliation_en': 'National Institute of Informatics',
    },
]



In [ ]:
if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
(len(idp_username_1), len(idp_password_1))

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# プロジェクトに対するMetadataアドオンの登録

- サブシステム名: アドオン
- ページ/アドオン: Metadata
- 機能分類: メタデータ入力
- シナリオ名: 書誌情報の入力
- 用意するテストデータ: DOI (10.52825/cordi.v1i.260)、URL一覧、アカウント(既存ユーザー1: GRDM)

## ウェブブラウザの新規プライベートウィンドウでGRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)
import scripts.metadata_v2025
importlib.reload(scripts.metadata_v2025)

from scripts.playwright import *
from scripts import grdm
from scripts.metadata_v2025 import ProjectMetadataForm, FileMetadataForm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)


In [ ]:
import time

async def _step(page):
    await page.goto(rdm_url)

    # 同意する をクリック
    await page.locator('//button[text() = "同意する"]').click()

    # 同意する が表示されなくなったことを確認
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step)

## ログイン情報を用いてGakuNin RDMにログインする

(IdPに関するログイン情報が与えられた場合、)
GakuNin Embeded DSのプルダウンを展開し、IdPリストから指定されたIdPを選択する。その後、アカウントのID/Passwordを入力して「Login」ボタンを押下する。

(IdPが指定されていない場合、)
CASのログイン操作を実施する。

In [ ]:
import scripts.grdm
importlib.reload(scripts.grdm)

async def _step(page):
    await scripts.grdm.login(
        page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout
    )

    # GRDMのボタンが表示されることを確認
    await expect(page.locator('//*[text() = "プロジェクト管理者"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクト一覧に指定されたタイトルのプロジェクトがない場合、指定された名前のプロジェクトを作成する

プロジェクト一覧に当該プロジェクト名が表示されていない場合、「新規プロジェクト作成」をクリックし、その名前を入力、「作成」をクリックする。

In [ ]:
import scripts.grdm
importlib.reload(scripts.grdm)

async def _step(page):
    await expect(page.locator('//*[@data-test-create-project-modal-button]')).to_have_count(1)

    await scripts.grdm.ensure_project_exists(page, rdm_project_name, transition_timeout=transition_timeout)
        
await run_pw(_step)

## ダッシュボードのプロジェクト一覧から指定されたプロジェクトをクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text()="{rdm_project_name}"]').click()        

    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

    await page.locator('//h3[text()="最近の活動"]').click()

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「アドオン」をクリックする

「アドオンを構成」のパネル内に「Metadata」の行が表示されること

In [ ]:
addon_name = 'Metadata'

async def _step(page):
    await page.locator('//a[text() = "アドオン"]').click()

    await page.locator(f'//div[@full_name = "{addon_name}"]').scroll_into_view_if_needed()
    await expect(page.locator(f'//h4[@class="addon-title"][normalize-space(.)="Metadata"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「メタデータ」をクリックする

メタデータの一覧ページが表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//a[contains(text(), "メタデータ")]').click()
    await expect(page.locator('//*[@data-test-new-metadata-button]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「新規メタデータ」をクリックする

スキーマ選択ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[@data-test-new-metadata-button]').click()
    
    await expect(page.locator('//*[@data-test-new-report-modal-schema="公的資金による研究データのメタデータ登録"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//*[@data-test-new-report-modal-create-report-button]')).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## 「メタデータを作成」をクリックする

メタデータ編集ウィンドウが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[@data-test-new-report-modal-create-report-button]').click()
    
    form = ProjectMetadataForm(page)
    await expect(form.get_locator('資金配分機関情報').locator('.ember-power-select-status-icon')).to_be_attached(timeout=transition_timeout)
    await expect(form.get_locator('体系的番号におけるプログラム情報コード')).to_be_editable(timeout=1000)
    await expect(form.get_locator('プログラム名 (日本語)')).to_be_editable(timeout=1000)
    await expect(form.get_locator('Program name (English)')).to_be_editable(timeout=1000)
    await expect(form.get_locator('体系的番号').locator('.ember-power-select-status-icon')).to_be_attached(timeout=1000)
    await expect(form.get_locator('プロジェクト名 (日本語)')).to_be_editable(timeout=1000)
    await expect(form.get_locator('Project name (English)')).to_be_editable(timeout=1000)
    await expect(form.get_locator('プロジェクトの分野').locator('.ember-power-select-status-icon')).to_be_attached(timeout=1000)

await run_pw(_step)


## 「資金配分機関情報」をクリックする

「国立研究開発法人科学技術振興機構 | JST」ほか選択肢が表示されること

In [ ]:
async def _step(page):
    form = ProjectMetadataForm(page)
    await form.get_locator('資金配分機関情報').locator('.ember-power-select-trigger').click()
    
    await expect(page.locator('//*[contains(@class, "ember-power-select-option") and text() = "国立研究開発法人科学技術振興機構 | JST"]')).to_be_attached(timeout=transition_timeout)

await run_pw(_step)


##  「国立研究開発法人科学技術振興機構 | JST」 をクリックする

「JST」が値に設定されること

In [ ]:
async def _step(page):
    form = ProjectMetadataForm(page)
    option = page.locator('//*[contains(@class, "ember-power-select-option") and text() = "国立研究開発法人科学技術振興機構 | JST"]')
    await expect(option).to_be_visible(timeout=transition_timeout)
    await option.click()
    
    await expect(form.get_locator('資金配分機関情報').locator('.ember-power-select-selected-item')).to_have_text('JST', timeout=transition_timeout)

await run_pw(_step)


## 「体系的番号におけるプログラム情報コード」に「ABC」を入力する

テキストフィールドに値が表示されること

In [ ]:
async def _step(page):
    form = ProjectMetadataForm(page)
    await form.fill('体系的番号におけるプログラム情報コード', 'ABC')
    
    await expect(form.get_locator('体系的番号におけるプログラム情報コード')).to_have_value('ABC', timeout=transition_timeout)

await run_pw(_step)


## 「プログラム名 (日本語)」に「サンプルプログラム」を入力する

テキストフィールドに値が表示されること

In [ ]:
async def _step(page):
    form = ProjectMetadataForm(page)
    await form.fill('プログラム名 (日本語)', 'サンプルプログラム')
    
    await expect(form.get_locator('プログラム名 (日本語)')).to_have_value('サンプルプログラム', timeout=transition_timeout)

await run_pw(_step)


## 「Program name (English)」に「Sample Program」を入力する

テキストフィールドに値が表示されること

In [ ]:
async def _step(page):
    form = ProjectMetadataForm(page)
    await form.fill('Program name (English)', 'Sample Program')
    
    await expect(form.get_locator('Program name (English)')).to_have_value('Sample Program', timeout=transition_timeout)

await run_pw(_step)


## 「体系的番号」をクリックする

テキストフィールドが表示されること

In [ ]:
async def _step(page):
    form = ProjectMetadataForm(page)
    await form.get_locator('体系的番号').locator('.ember-power-select-trigger').click()
    
    await expect(page.locator('//input[contains(@class, "ember-power-select-search-input")]')).to_be_editable(timeout=transition_timeout)

await run_pw(_step)


##  「0123456789(ENTER)」 を入力する

「0123456789」が値に設定されること

In [ ]:
async def _step(page):
    await page.locator('//input[contains(@class, "ember-power-select-search-input")]').fill('0123456789')
    await page.keyboard.press('Enter')
    
    form = ProjectMetadataForm(page)
    await expect(form.get_locator('体系的番号').locator('.ember-power-select-selected-item')).to_have_text('0123456789', timeout=transition_timeout)

await run_pw(_step)


## 「プロジェクト名 (日本語)」に「サンプルメタデータプロジェクト」を入力する

テキストフィールドに値が表示されること

In [ ]:
async def _step(page):
    form = ProjectMetadataForm(page)
    await form.fill('プロジェクト名 (日本語)', 'サンプルメタデータプロジェクト')
    
    await expect(form.get_locator('プロジェクト名 (日本語)')).to_have_value('サンプルメタデータプロジェクト', timeout=transition_timeout)

await run_pw(_step)


## 「Project name (English)」に「Sample Metadata Project」を入力する

テキストフィールドに値が表示されること

In [ ]:
async def _step(page):
    form = ProjectMetadataForm(page)
    await form.fill('Project name (English)', 'Sample Metadata Project')
    
    await expect(form.get_locator('Project name (English)')).to_have_value('Sample Metadata Project', timeout=transition_timeout)

await run_pw(_step)


## 「プロジェクトの分野」をクリックする

テキストフィールドが表示されること

In [ ]:
async def _step(page):
    form = ProjectMetadataForm(page)
    await form.get_locator('プロジェクトの分野').locator('.ember-power-select-trigger').click()
    
    await expect(page.locator('//*[@data-test-option = "情報通信 | 289"]')).to_be_attached(timeout=transition_timeout)

await run_pw(_step)


##  「情報通信 | 289」 をクリックする

「289」が値に設定されること

In [ ]:
async def _step(page):
    await page.locator('//*[@data-test-option = "情報通信 | 289"]').click()
    
    form = ProjectMetadataForm(page)
    await expect(form.get_locator('プロジェクトの分野').locator('.ember-power-select-selected-item')).to_have_text('289', timeout=transition_timeout)

await run_pw(_step)


## 「次へ」をクリックする

左側ペイン「メタデータ登録」が緑色かつチェックマークが表示されること

In [ ]:
import re

async def _step(page):
    await page.locator('//*[@data-test-goto-next-page]').click()
    
    await expect(page.locator('//span[@data-test-label and text() = "メタデータ登録"]/../preceding-sibling::i')).to_have_class(re.compile(r'.*fa-check-circle-o.*'), timeout=transition_timeout)

await run_pw(_step)

## 「内容確認」をクリックする

左側ペイン「登録データ」が赤色かつ！マークが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[@data-test-goto-review]').click()
    
    await expect(page.locator('//span[@data-test-label and text() = "登録データ"]/../preceding-sibling::i')).to_have_class(re.compile(r'.*fa-exclamation-circle.*'), timeout=transition_timeout)

await run_pw(_step)

## 「登録データ一覧」編集ボタンをクリックする

「登録データ」編集画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[@aria-label="Edit 登録データ一覧|Registered Data List"]').click()
    
    await expect(page.locator(f'//a[starts-with(@href, "{rdm_url}") and @target = "_blank"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「登録データ一覧」の説明中のURLをクリックする

プロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    popup_future = page.wait_for_event('popup')
    await page.locator(f'//a[starts-with(@href, "{rdm_url}") and @target = "_blank"]').click()
    popup = await popup_future
    
    await expect(popup.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=30000)
    await expect(grdm.get_select_expanded_storage_title_locator(popup, target_storage_name)).to_be_visible(timeout=transition_timeout)
    time.sleep(1)
    return popup

await run_pw(_step)

## 対象ストレージをクリックする

「メタデータ編集」ボタンが表示されること (表示まで数秒かかる可能性がある)

In [ ]:
async def _step(page):
    await grdm.get_select_expanded_storage_title_locator(page, target_storage_name).click()
    
    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「新規フォルダ作成」をクリックする

フォルダ名入力テキストフィールドが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "新規フォルダ作成"]').click()
    
    await expect(page.locator('//input[@id = "createFolderInput"]')).to_be_editable(timeout=transition_timeout)

await run_pw(_step)

## 「TESTMETADATA(ENTER)」を入力する

TESTMETADATAフォルダが作成されること

In [ ]:
async def _step(page):
    await page.locator('//input[@id = "createFolderInput"]').fill('TESTMETADATA')
    await page.keyboard.press('Enter')

    await expect(page.locator('//*[text() = "TESTMETADATA"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「TESTMETADATA」をクリックする

「メタデータ編集」ボタンが表示されること (表示まで数秒かかる可能性がある)

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "TESTMETADATA"]').click()

    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

##  「メタデータ編集」をクリックする

「メタデータ編集」ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "メタデータ編集"]').click()

    await expect(page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select')).to_be_editable(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## 「ファイル種別」で「論文」を選択する

論文専用項目が表示されること

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    locator = form.get_locator('ファイル種別')
    await locator.select_option('manuscript')
    await expect(locator).to_have_value('manuscript', timeout=transition_timeout)

await run_pw(_step)


## 「データの名称(日本語)」に論文タイトルを入力する

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await form.fill('データの名称または論文表題 (日本語)', paper_title_ja)
    await expect(form.get_locator('データの名称または論文表題 (日本語)')).to_have_value(paper_title_ja, timeout=transition_timeout)

await run_pw(_step)


## 「Title (English)」に論文タイトルを入力する

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await form.fill('Title (English)', paper_title_en)
    await expect(form.get_locator('Title (English)')).to_have_value(paper_title_en, timeout=transition_timeout)

await run_pw(_step)


## 「論文（出版社版）のDOI」に DOI を入力する

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await form.fill('論文（出版社版）のDOI', paper_doi)
    await page.keyboard.press('Tab')
    await expect(form.get_locator('論文（出版社版）のDOI')).to_have_value(paper_doi, timeout=transition_timeout)

await run_pw(_step)


## 「論文の種類」で「学術雑誌論文」を選択する

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    locator = form.get_locator('論文の種類')
    await locator.select_option('journal article')
    await expect(locator).to_have_value('journal article', timeout=transition_timeout)

await run_pw(_step)


## 「著者名」を入力する

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    for author in paper_authors:
        await form.fill_author(author)

await run_pw(_step)


## 「掲載誌名 (日本語)」に掲載誌名を入力する

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await form.fill('掲載誌名 (日本語)', paper_journal_name_ja)
    await page.keyboard.press('Tab')
    await expect(form.get_locator('掲載誌名 (日本語)')).to_have_value(paper_journal_name_ja, timeout=transition_timeout)

await run_pw(_step)


## 「Journal Name (English)」に掲載誌名を入力する

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await form.fill('Journal Name (English)', paper_journal_name_en)
    await page.keyboard.press('Tab')
    await expect(form.get_locator('Journal Name (English)')).to_have_value(paper_journal_name_en, timeout=transition_timeout)

await run_pw(_step)


## 「発行年月」に公開年月を入力する

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await form.fill('発行年月', paper_publication_month)
    await page.keyboard.press('Tab')
    await expect(form.get_locator('発行年月')).to_have_value(paper_publication_month, timeout=transition_timeout)

await run_pw(_step)


## 「巻」に「1」を入力する

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await form.fill('巻', paper_journal_volume)
    await page.keyboard.press('Tab')
    await expect(form.get_locator('巻')).to_have_value(paper_journal_volume, timeout=transition_timeout)

await run_pw(_step)


## 「号」に「1」を入力する

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await form.fill('号', paper_journal_issue)
    await page.keyboard.press('Tab')
    await expect(form.get_locator('号')).to_have_value(paper_journal_issue, timeout=transition_timeout)

await run_pw(_step)


## 「掲載ページ (開始)」にページ番号を入力する


In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await form.fill('掲載ページ (開始)', paper_page_start)
    await page.keyboard.press('Tab')
    await expect(form.get_locator('掲載ページ (開始)')).to_have_value(paper_page_start, timeout=transition_timeout)

await run_pw(_step)


## 「掲載ページ (終了)」にページ番号を入力する


In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await form.fill('掲載ページ (終了)', paper_page_end)
    await page.keyboard.press('Tab')
    await expect(form.get_locator('掲載ページ (終了)')).to_have_value(paper_page_end, timeout=transition_timeout)

await run_pw(_step)


## 「査読の有無」で「あり」を選択する


In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    locator = form.get_locator('査読の有無')
    await locator.select_option('yes')
    await expect(locator).to_have_value('yes', timeout=transition_timeout)

await run_pw(_step)


## 「版情報」で「著者最終稿」を選択する


In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    locator = form.get_locator('版情報')
    await locator.select_option('AM')
    await expect(locator).to_have_value('AM', timeout=transition_timeout)

await run_pw(_step)


## 「備考 (日本語)」に補足を入力する

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await form.fill('備考 (日本語)', '書誌情報の入力を行いました。')
    await page.keyboard.press('Tab')
    await expect(form.get_locator('備考 (日本語)')).to_have_value('書誌情報の入力を行いました。', timeout=transition_timeout)

await run_pw(_step)


## 「Remarks (English)」に補足を入力する

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await form.fill('Remarks (English)', 'Bibliographic metadata entry completed.')
    await page.keyboard.press('Tab')
    await expect(form.get_locator('Remarks (English)')).to_have_value('Bibliographic metadata entry completed.', timeout=transition_timeout)

await run_pw(_step)


## 「保存」をクリックする

In [ ]:
async def _step(page):
    await page.locator('//a[text() = "保存"]').click()
    await expect(page.locator('//a[text() = "保存"]').first).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「TESTMETADATA」をクリックする

「メタデータ登録」ボタンが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "TESTMETADATA"]').click()
    await expect(page.locator('//*[text() = "メタデータ登録"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)


## 「メタデータ登録」をクリックする

「ファイルメタデータ登録先の選択」ダイアログが表示され、「サンプルメタデータプロジェクト」とチェックボックスが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "メタデータ登録"]').click()
    await expect(page.locator('//input[starts-with(@id, "draft-")]')).to_have_count(1, timeout=transition_timeout)

await run_pw(_step)


## 「サンプルメタデータプロジェクト」をクリックする

チェックボックスがチェック状態になること

In [ ]:
async def _step(page):
    checkbox = page.locator('//input[starts-with(@id, "draft-")]')
    await checkbox.click()
    await expect(checkbox).to_be_checked(timeout=transition_timeout)

await run_pw(_step)


## 「選択」をクリックする

「サンプルメタデータプロジェクト」に「開く」リンクが現れること

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "選択"]').click()
    await expect(page.locator('//*[text() = "開く"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)


## 「開く」をクリックする

ワークフローメタデータ編集画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "開く"]').click()
    await expect(page.locator('//*[@data-test-goto-review]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「内容確認」をクリックする

左側ペイン「登録データ」が緑色かつチェックマークが表示されること

In [ ]:
import re

async def _step(page):
    await page.locator('//*[@data-test-goto-review]').click()
    await expect(page.locator('//span[@data-test-label and text() = "登録データ"]/../preceding-sibling::i')).to_have_class(re.compile(r'.*fa-check-circle-o.*'), timeout=transition_timeout)

await run_pw(_step)


## 「エクスポート」をクリックする

言語選択画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//button[@data-test-registration-card-export]').click()
    await expect(page.locator('//select[@id = "registration-report-format-selection"]')).to_be_enabled(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)


## 「メタデータ共通項目2024版CSV形式 (日本語)」を選択し「エクスポート」をクリックする

CSVファイルがダウンロードされ、UTF-16 BOM付きでヘッダーのみ（データ行なし）となること。書誌情報用テンプレートでは項目が出力されないことを確認する

In [ ]:
import os

async def _step(page):
    await page.locator('//select[@id = "registration-report-format-selection"]').select_option('メタデータ共通項目2024版CSV形式 (日本語)')
    download_future = page.wait_for_event('download')
    await page.locator('//button[@data-test-registration-report-submit]').click()
    download = await download_future
    csv_path = os.path.join(work_dir, 'report.csv')
    await download.save_as(csv_path)
    assert os.path.exists(csv_path)
    df = pd.read_csv(csv_path, encoding='utf-16')
    expected_columns = [
        '資金配分機関情報',
        '体系的番号におけるプログラム情報コード',
        'プログラム名',
        '体系的番号',
        'プロジェクト名',
        'データNo.',
        'データの名称',
        '掲載日・掲載更新日',
        'データの説明',
        'データの分野',
        'データ種別',
        '概略データ量',
        '管理対象データの利活用・提供方針 (有償/無償)',
        '管理対象データの利活用・提供方針(ライセンス)',
        '管理対象データの利活用・提供方針(引用方法等)',
        'アクセス権',
        '公開予定日 (公開期間猶予の場合)',
        'リポジトリ情報',
        'リポジトリURL・DOIリンク',
        'データ作成者',
        'データ作成者の研究者番号',
        'データ管理機関',
        'データ管理機関コード',
        'データ管理者',
        'データ管理者の研究者番号',
        'データ管理者の所属組織名',
        'データ管理者の所属機関の連絡先住所',
        'データ管理者の所属機関の連絡先電話番号',
        'データ管理者の所属機関の連絡先メールアドレス',
        '備考'
    ]
    assert list(df.columns) == expected_columns, df.columns
    assert df.empty, df

await run_pw(_step)


In [ ]:
!rm {work_dir}/report.csv


## 「エクスポート」をクリックする

言語選択画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//button[@data-test-registration-card-export]').click()
    await expect(page.locator('//select[@id = "registration-report-format-selection"]')).to_be_enabled(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)


## 「Common Metadata Elements 2024 edition CSV format (English)」を選択し「エクスポート」をクリックする

CSVファイルがダウンロードされ、UTF-16 BOM付きでヘッダーのみ（データ行なし）となること。書誌情報用テンプレートでは項目が出力されないことを確認する

In [ ]:
import os

async def _step(page):
    await page.locator('//select[@id = "registration-report-format-selection"]').select_option('Common Metadata Elements 2024 edition CSV format (English)')
    download_future = page.wait_for_event('download')
    await page.locator('//button[@data-test-registration-report-submit]').click()
    download = await download_future
    csv_path = os.path.join(work_dir, 'report.csv')
    await download.save_as(csv_path)
    assert os.path.exists(csv_path)
    df = pd.read_csv(csv_path, encoding='utf-16')
    expected_columns = [
        'Funder',
        'Funding stream code in Japan Grant Number',
        'Program name',
        'Japan Grant Number',
        'Project name',
        'Data No.',
        'Title',
        'Date (Issued / Updated)',
        'Description',
        'Research field',
        'Data type',
        'File size',
        'Data utilization and provision policy(Free/Consideration)',
        'Data utilization and provision policy(License)',
        'Data utilization and provision policy(citation information)',
        'Access rights',
        'Publication date (for embargoed access)',
        'Repository information',
        'Repository URL/ DOI link',
        'Creator Name',
        'Creator name identifier (e-Rad)',
        'Hosting institution',
        'Hosting institution Identifier',
        'Data manager',
        'Data manager identifier (e-Rad)',
        'Contact organization of data manager',
        'Contact address of data manager',
        'Contact phone number of data manager',
        'Contact mail address of data manager',
        'Remarks'
    ]
    assert list(df.columns) == expected_columns, df.columns
    assert df.empty, df

await run_pw(_step)


In [ ]:
!rm {work_dir}/report.csv


## RDMのURL `/search/` にアクセスする

検索窓が表示されること

In [ ]:
async def _step(page):
    _rdm_url = rdm_url[:-1] if rdm_url.endswith('/') else rdm_url
    await page.goto(f'{_rdm_url}/search/')
    await expect(page.locator('#searchPageFullBar')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)


## メタデータタイトルに入れた検索キーワードを入力し、Enterキーを押す

1つだけ、ファイルメタデータが検索結果に表示されること

In [ ]:
async def _step(page):
    await page.locator('#searchPageFullBar').fill(metadata_search_keyword)
    await page.keyboard.press('Enter')
    await expect(page.locator(f'//a[text() = "{target_storage_id}/TESTMETADATA/"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## ファイルメタデータのリンクをクリックする

メタデータ編集画面が表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//a[text() = "{target_storage_id}/TESTMETADATA/"]').click()
    await expect(page.locator('//label[contains(text(), "データの名称または論文表題 (日本語)")]/../following-sibling::div[1]//input')).to_have_value(paper_title_ja, timeout=transition_timeout)

await run_pw(_step)


## 「閉じる」をクリックする

ファイル一覧画面が表示されること

In [ ]:
import asyncio

async def _step(page):
    await page.locator('//*[@class = "modal-footer"]//a[text() = "閉じる"]').click()
    await expect(page.locator('//*[text() = "メタデータ登録"]')).to_be_enabled(timeout=transition_timeout)
    await asyncio.sleep(1)

await run_pw(_step)


## 「メタデータ削除」をクリックする

メタデータを削除してよろしいですか？この操作は元に戻せません。 と表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "メタデータ削除"]').click()
    await expect(page.locator('//*[text() = "メタデータを削除してよろしいですか？この操作は元に戻せません。"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(1)

await run_pw(_step)

## 「削除」をクリックする

In [ ]:
file_view_url = None
async def _step(page):
    await page.locator('//*[contains(@class, "btn-danger") and text() = "削除"]').click()

    await expect(page.locator('//*[text() = "メタデータを削除してよろしいですか？この操作は元に戻せません。"]')).not_to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(1)

    # 次以降の検索トライアル後に戻って来れるように、URLを控えておく
    global file_view_url
    file_view_url = page.url
    print(f'file_view_url: {file_view_url}')

await run_pw(_step)

## RDMのURL `/search/` にアクセスする

検索窓が表示されること

In [ ]:
async def _step(page):
    _rdm_url = rdm_url[:-1] if rdm_url.endswith('/') else rdm_url
    await page.goto(f'{_rdm_url}/search/')

    await expect(page.locator('#searchPageFullBar')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## メタデータタイトルに入れた検索キーワードを入力し、Enterキーを押す

ファイルメタデータが検索結果に表示されないこと

In [ ]:
async def _step(page):
    await page.locator('#searchPageFullBar').fill(metadata_search_keyword)
    await page.keyboard.press('Enter')

    await asyncio.sleep(10)
    await expect(page.locator(f'//a[text() = "{target_storage_id}/TESTMETADATA/"]')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## ブラウザの戻るボタンをクリックする

ファイル一覧画面が表示され、 TESTMETADATA が選択状態であること

In [ ]:
async def _step(page):
    await page.goto(file_view_url)

    await expect(page.locator('//*[contains(@class, "fangorn-selected")]//*[text() = "TESTMETADATA"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)    

## (指定がある場合) プロジェクトを削除する

In [ ]:
async def _step(page):
    if not delete_project:
        return
    await scripts.grdm.delete_project(page)
    
    await expect(page.locator('//*[text() = "プロジェクト管理者"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

終了処理を実施。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}